# EDA_Analysis — Mutual Fund Analytics
This notebook compiles **15+ EDA charts** using processed datasets under `Data/processed`.
Charts are also exported as PNGs to `reports/eda_png/` when executed.

In [2]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio

sns.set_theme(style='whitegrid', context='talk')

# --- repo root detection ---
_HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()
def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(12):
        if (cand / 'Data' / 'processed').exists():
            return cand
        cand = cand.parent
    return start.parent

_REPO_ROOT = _find_repo_root(_HERE)
DATA_DIR = _REPO_ROOT / 'Data' / 'processed'
OUT_DIR = _REPO_ROOT / 'reports' / 'eda_png'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR:', DATA_DIR)
print('OUT_DIR:', OUT_DIR)

# Helper: matplotlib save
def save_matplotlib(fig, name: str):
    path = OUT_DIR / f'{name}.png'
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return path

# Helper: plotly save
def save_plotly(fig, name: str):
    path = OUT_DIR / f'{name}.png'
    # Requires kaleido for static export; fall back to HTML if missing
    try:
        pio.write_image(fig, str(path), scale=2)
        return path
    except Exception as e:
        html_path = OUT_DIR / f'{name}.html'
        fig.write_html(str(html_path))
        print('Plotly PNG export failed, wrote HTML instead:', e)
        return html_path


DATA_DIR: c:\Mutual Fund Analytics\Data\processed
OUT_DIR: c:\Mutual Fund Analytics\reports\eda_png


In [3]:
# Load core processed datasets
def read_processed(fname: str) -> pd.DataFrame:
    p = DATA_DIR / fname
    if not p.exists():
        raise FileNotFoundError(p)
    return pd.read_csv(p)

fund_master = read_processed('fund_master_clean.csv')
nav = read_processed('nav_history_clean.csv')
aum = read_processed('aum_by_fund_house_clean.csv')
industry_folio = read_processed('industry_folio_count_clean.csv')
monthly_sip = read_processed('monthly_sip_inflows_clean.csv')
category_inflows = read_processed('category_inflows_clean.csv')
portfolio_holdings = read_processed('portfolio_holdings_clean.csv')
scheme_perf = read_processed('scheme_performance_clean.csv')
investor_tx = read_processed('investor_transactions_clean.csv')
print('Loaded datasets:', {
    'fund_master': fund_master.shape,
    'nav': nav.shape,
    'aum': aum.shape,
    'industry_folio': industry_folio.shape,
    'monthly_sip': monthly_sip.shape,
    'category_inflows': category_inflows.shape,
    'portfolio_holdings': portfolio_holdings.shape,
    'scheme_perf': scheme_perf.shape,
    'investor_tx': investor_tx.shape
})

Loaded datasets: {'fund_master': (40, 15), 'nav': (64320, 3), 'aum': (90, 5), 'industry_folio': (21, 6), 'monthly_sip': (48, 6), 'category_inflows': (144, 3), 'portfolio_holdings': (322, 8), 'scheme_perf': (40, 20), 'investor_tx': (32778, 13)}


## Charts (15+)

In [4]:
# 1) Industry folio count growth line (reuses the idea from the dedicated notebook)
industry_folio['month'] = pd.to_datetime(industry_folio['month'], errors='coerce')
industry_folio['total_folios_crore'] = pd.to_numeric(industry_folio['total_folios_crore'], errors='coerce')
df1 = industry_folio.dropna(subset=['month','total_folios_crore']).sort_values('month')

fig = go.Figure()
fig.add_trace(go.Scatter(x=df1['month'], y=df1['total_folios_crore'], mode='lines+markers', name='Total folios (Cr)'))
fig.update_layout(title='Industry folio count growth', template='plotly_white', xaxis_title='Month', yaxis_title='Folio count (Cr)')
save_plotly(fig, '01_industry_folio_growth')
fig.show()

Plotly PNG export failed, wrote HTML instead: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



In [34]:
# 2) Stacked area of folio categories over time
cols = ['equity_folios_crore','debt_folios_crore','hybrid_folios_crore','others_folios_crore']
for c in cols:
    industry_folio[c] = pd.to_numeric(industry_folio[c], errors='coerce')
df2 = industry_folio.dropna(subset=['month']+cols).copy()
df2 = df2.sort_values('month')

plt.figure(figsize=(12,6))
plt.stackplot(df2['month'], [df2[c] for c in cols], labels=cols)
plt.legend(loc='upper left')
plt.title('Industry folios — category mix')
plt.xlabel('Month')
plt.ylabel('Folios (Cr)')
fig = plt.gcf()
save_matplotlib(fig, '02_industry_folio_category_mix')
plt.show()

In [35]:
# 3) AUM by fund house (top 10 at latest date)
aum['date'] = pd.to_datetime(aum['date'], errors='coerce')
aum['aum_lakh_crore'] = pd.to_numeric(aum['aum_lakh_crore'], errors='coerce')
latest = aum.sort_values('date').groupby('fund_house').tail(1)
top = latest.sort_values('aum_lakh_crore', ascending=False).head(10)
top = top.sort_values('aum_lakh_crore')

fig = plt.figure(figsize=(10,6))
plt.barh(top['fund_house'], top['aum_lakh_crore'])
plt.title('Top 10 fund houses by AUM (latest snapshot)')
plt.xlabel('AUM (Lakh Cr)')
plt.tight_layout()
save_matplotlib(fig, '03_top10_fundhouses_aum')
plt.show()

In [36]:
# 4) AUM trend for one dominant fund house (SBI Mutual Fund if present)
SBI_NAME = 'SBI Mutual Fund'
if SBI_NAME in aum['fund_house'].unique():
    sbi = aum[aum['fund_house']==SBI_NAME].sort_values('date')
    fig = plt.figure(figsize=(12,6))
    plt.plot(sbi['date'], sbi['aum_lakh_crore'])
    plt.title('AUM trend — SBI Mutual Fund')
    plt.xlabel('Date')
    plt.ylabel('AUM (Lakh Cr)')
    save_matplotlib(fig, '04_aum_trend_sbi')
    plt.show()
else:
    print('SBI Mutual Fund not found; skipping chart 04')

In [37]:
# 5) Monthly SIP inflows (industry)
monthly_sip['year_month'] = pd.to_datetime(monthly_sip['year_month'], errors='coerce') if 'year_month' in monthly_sip.columns else pd.to_datetime(monthly_sip['month'], errors='coerce')
inflow_col = 'sip_inflow_crore' if 'sip_inflow_crore' in monthly_sip.columns else 'total_sip_inflow_crore'
monthly_sip[inflow_col] = pd.to_numeric(monthly_sip[inflow_col], errors='coerce')
df5 = monthly_sip.dropna(subset=['year_month', inflow_col]).sort_values('year_month')
fig = go.Figure()
fig.add_trace(go.Scatter(x=df5['year_month'], y=df5[inflow_col], mode='lines+markers', name='SIP inflow (Cr)'))
fig.update_layout(title='Monthly SIP inflows — industry', template='plotly_white', xaxis_title='Month', yaxis_title='SIP inflow (Cr)')
save_plotly(fig, '05_monthly_sip_industry')
fig.show()

Plotly PNG export failed, wrote HTML instead: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



In [38]:
# 6) Category inflows heatmap (year_month x category) if columns exist
# Expecting category_inflows_clean.csv schema: year_month, category, inflow_crore (or similar)
cols = set(category_inflows.columns)
ym_col = 'year_month' if 'year_month' in cols else ('month' if 'month' in cols else None)
cat_col = 'category' if 'category' in cols else None
val_col = None
for candidate in ['category_inflow_crore','inflow_crore','sip_inflow_crore','net_inflow_crore','inflow']:
    if candidate in cols:
        val_col = candidate
        break

if ym_col and cat_col and val_col:
    tmp = category_inflows.copy()
    tmp[ym_col] = pd.to_datetime(tmp[ym_col], errors='coerce')
    tmp[val_col] = pd.to_numeric(tmp[val_col], errors='coerce')
    pivot = tmp.pivot_table(index=tmp[ym_col].dt.year.astype(str)+'-'+tmp[ym_col].dt.month.astype(str).str.zfill(2), columns=cat_col, values=val_col, aggfunc='sum')
    pivot = pivot.sort_index().fillna(0)
    fig = plt.figure(figsize=(14,7))
    sns.heatmap(pivot.T, cmap='YlGnBu')
    plt.title('Category inflows heatmap')
    plt.xlabel('Month')
    plt.ylabel('Category')
    save_matplotlib(fig, '06_category_inflows_heatmap')
    plt.show()
else:
    print('Category inflows columns not compatible; skipping chart 06')
    

In [39]:
# 7) Sector allocation donut (aggregate equity sectors from portfolio holdings)
# Expect: sector, weight_pct, scheme type filtering not provided; use weight_pct sum across all records
ph = portfolio_holdings.copy()
if 'weight_pct' in ph.columns and 'sector' in ph.columns:
    ph['weight_pct'] = pd.to_numeric(ph['weight_pct'], errors='coerce')
    sec = ph.dropna(subset=['sector','weight_pct']).groupby('sector', as_index=False)['weight_pct'].sum()
    sec = sec.sort_values('weight_pct', ascending=False)
    fig = go.Figure(go.Pie(labels=sec['sector'], values=sec['weight_pct'], hole=0.45))
    fig.update_layout(title='Sector allocation (aggregate holdings)')
    save_plotly(fig, '07_sector_allocation_donut')
    fig.show()
else:
    print('portfolio_holdings missing sector/weight_pct; skipping chart 07')

Plotly PNG export failed, wrote HTML instead: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



In [40]:
# 8) NAV distribution histogram for latest date
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')
nav['nav'] = pd.to_numeric(nav['nav'], errors='coerce')
latest_date = nav['date'].max()
df8 = nav[nav['date']==latest_date].dropna(subset=['nav'])
fig = plt.figure(figsize=(10,6))
sns.histplot(df8['nav'], bins=40, kde=False)
plt.title(f'NAV distribution (latest date: {latest_date.date()})')
plt.xlabel('NAV')
plt.ylabel('Count')
save_matplotlib(fig, '08_nav_distribution')
plt.show()

In [41]:
# 9) Top 10 schemes by 1-year return (from scheme_performance)
if 'return_1yr_pct' in scheme_perf.columns and 'scheme_name' in scheme_perf.columns:
    sp = scheme_perf.copy()
    sp['return_1yr_pct'] = pd.to_numeric(sp['return_1yr_pct'], errors='coerce')
    sp = sp.dropna(subset=['return_1yr_pct'])
    top = sp.sort_values('return_1yr_pct', ascending=False).head(10)
    fig = plt.figure(figsize=(12,6))
    sns.barplot(data=top, x='return_1yr_pct', y='scheme_name')
    plt.title('Top 10 schemes by 1Y return')
    save_matplotlib(fig, '09_top10_1yr_returns')
    plt.show()
else:
    print('scheme_performance columns not compatible; skipping chart 09')

In [42]:
# 10) Correlation heatmap of returns among 10 selected funds using scheme_performance_daily if available
# Fallback: use NAV-derived daily returns for 10 schemes.

# pick 10 schemes with enough history
amfi_col = 'amfi_code' if 'amfi_code' in nav.columns else ('scheme_amfi_code' if 'scheme_amfi_code' in nav.columns else None)
if amfi_col and 'scheme_name' in fund_master.columns:
    # choose top 10 codes by number of nav rows
    code_counts = nav[amfi_col].value_counts().head(10).index.tolist()
    n10 = nav[nav[amfi_col].isin(code_counts)].copy().sort_values(['amfi_code','date'])
    n10['nav'] = pd.to_numeric(n10['nav'], errors='coerce')
    # daily pct return per scheme
    n10['ret'] = n10.groupby(amfi_col)['nav'].pct_change()
    rets = n10.pivot_table(index='date', columns=amfi_col, values='ret', aggfunc='mean')
    corr = rets.corr()
    fig = plt.figure(figsize=(12,10))
    sns.heatmap(corr, cmap='coolwarm', center=0)
    plt.title('NAV return correlation — 10 selected schemes')
    save_matplotlib(fig, '10_nav_return_correlation_heatmap')
    plt.show()
else:
    print('NAV correlation prerequisites not met; skipping chart 10')

In [43]:
# 11) Investor transactions: transaction type counts
tx = investor_tx.copy()
if 'transaction_type' in tx.columns:
    fig = plt.figure(figsize=(10,6))
    sns.countplot(data=tx, x='transaction_type')
    plt.title('Investor transactions by type')
    save_matplotlib(fig, '11_tx_type_counts')
    plt.show()
else:
    print('transaction_type column missing; skipping chart 11')

In [44]:
# 12) SIP inflow all-time high annotation (from monthly_sip if possible)
if inflow_col in monthly_sip.columns:
    tmp = monthly_sip.copy()
    tmp['year_month'] = pd.to_datetime(tmp['year_month'] if 'year_month' in tmp.columns else tmp['month'], errors='coerce')
    tmp[inflow_col] = pd.to_numeric(tmp[inflow_col], errors='coerce')
    idx = tmp[inflow_col].idxmax()
    high_row = tmp.loc[idx]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=tmp['year_month'], y=tmp[inflow_col], mode='lines', name='SIP inflow'))
    fig.add_trace(go.Scatter(x=[high_row['year_month']], y=[high_row[inflow_col]], mode='markers+text', text=['All-time high'], textposition='top center'))
    fig.update_layout(title='SIP inflows — all-time high', template='plotly_white', xaxis_title='Month', yaxis_title='SIP inflow (Cr)')
    save_plotly(fig, '12_sip_all_time_high')
    fig.show()
else:
    print('SIP inflow column missing; skipping chart 12')

Plotly PNG export failed, wrote HTML instead: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



In [45]:
# 13) NAV vs date for a few selected schemes
if amfi_col:
    selected = nav[amfi_col].value_counts().head(3).index.tolist()
    fig = go.Figure()
    for code in selected:
        d = nav[nav[amfi_col]==code].copy().sort_values('date')
        fig.add_trace(go.Scatter(x=d['date'], y=d['nav'], mode='lines', name=str(code)))
    fig.update_layout(title='NAV history — 3 selected schemes', template='plotly_white', xaxis_title='Date', yaxis_title='NAV')
    save_plotly(fig, '13_nav_history_selected_schemes')
    fig.show()
else:
    print('amfi_code column missing; skipping chart 13')

Plotly PNG export failed, wrote HTML instead: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



In [ ]:
# 14) Portfolio holdings top sectors by average weight
ph = portfolio_holdings.copy()
if 'sector' in ph.columns and 'weight_pct' in ph.columns:
    ph['weight_pct'] = pd.to_numeric(ph['weight_pct'], errors='coerce')
    ph['sector'] = ph['sector'].astype(str)
    sec_avg = ph.dropna(subset=['sector','weight_pct']).groupby('sector', as_index=False)['weight_pct'].mean().sort_values('weight_pct', ascending=False).head(10)
    fig = plt.figure(figsize=(12,6))
    sns.barplot(data=sec_avg, x='weight_pct', y='sector')
    plt.title('Top 10 sectors by average weight_pct')
    save_matplotlib(fig, '14_top10_sectors_avg_weight')
    plt.show()
else:
    print('portfolio_holdings missing sector/weight_pct; skipping chart 14')

In [ ]:
# 15) Scheme category distribution (counts)
if 'category' in fund_master.columns:
    fig = plt.figure(figsize=(12,6))
    top = fund_master['category'].astype(str).value_counts().head(15)
    sns.barplot(x=top.values, y=top.index)
    plt.title('Scheme category distribution (top 15)')
    plt.xlabel('Number of schemes')
    plt.ylabel('Category')
    save_matplotlib(fig, '15_scheme_category_distribution')
    plt.show()
else:
    print('fund_master missing category; skipping chart 15')

## Outputs
Run cells to render charts and export PNGs to:
- `reports/eda_png/`